# Chromatic Parallel Metropolis for the Antiferromagnetic Potts Model

This notebook explains the **chromatic-parallel part** of our project proposal.

Our model is
$$
\text{Potts energy}
+
\text{Boltzmann distribution}
+
\text{Metropolis accept/reject}
+
\text{chromatic parallel schedule}
$$

We use $q$ Potts color labels. We do **not** use a two-state Ising spin model, and the local move below is a **Metropolis move**.

# 1. State of the Graph

Let the graph be a finite simple graph
$$
G=(V,E),
\qquad
n=|V|
$$

Each vertex has one of $q$ possible color labels:
$$
\sigma_v\in\{1,2,\ldots,q\},
\qquad q\ge 3
$$

The whole coloring is
$$
\sigma=(\sigma_v)_{v\in V}
$$

The state space is
$$
\Omega=\{1,2,\ldots,q\}^{V}
$$

An edge $\{u,v\}$ is a **conflict** when
$$
\sigma_u=\sigma_v
$$

These are Potts color labels, not spins in $\{-1,+1\}$.

# 2. Antiferromagnetic Potts Energy

Define the conflict indicator
$$
\delta(\sigma_u,\sigma_v)
=
\begin{cases}
1, & \sigma_u=\sigma_v,\\
0, & \sigma_u\ne\sigma_v.
\end{cases}
$$

Our energy function is
$$
H(\sigma)
=
J\sum_{\{u,v\}\in E}
\delta(\sigma_u,\sigma_v),
\qquad J>0
$$

Every conflicting edge adds the positive penalty $J$.

If $J=1$, then
$$
H(\sigma)=\#\text{ conflicting edges}
$$

A valid $q$-coloring has
$$
H(\sigma)=0
$$

This zero-energy state exists only when $G$ is $q$-colorable. Otherwise the smallest possible energy is greater than zero.

This is the antiferromagnetic convention used for the graph-coloring objective: neighboring vertices are encouraged to have different labels.

# 3. Boltzmann Distribution

The probability of a coloring is
$$
\pi(\sigma)
=
\frac{e^{-\beta H(\sigma)}}{Z}
$$

where
$$
Z
=
\sum_{\tau\in\Omega}e^{-\beta H(\tau)}
$$

$Z$ is the normalization factor.

$\beta$ is the inverse temperature:
$$
\beta=\frac{1}{T}
$$

The probability ratio between a proposed coloring $\sigma'$ and the current coloring $\sigma$ is
$$
\frac{\pi(\sigma')}{\pi(\sigma)}
=
e^{-\beta[H(\sigma')-H(\sigma)]}
=
e^{-\beta\Delta H}
$$

The factor $Z$ cancels, so it does not need to be calculated for a Metropolis move.

# 4. One-Vertex Metropolis Proposal

Choose a vertex $v$.

Let its current Potts label be
$$
a=\sigma_v
$$

Propose a different label
$$
b\in\{1,2,\ldots,q\}\setminus\{a\}
$$

For a uniform proposal,
$$
Q_v(b\mid a)=\frac{1}{q-1}
$$

This proposal is symmetric because
$$
Q_v(b\mid a)=Q_v(a\mid b)
$$

Only the label at vertex $v$ is proposed to change. Every other vertex keeps its current label.

# 5. Local Energy Change

Let $\mathcal N(v)$ be the neighbors of vertex $v$.

This function counts how many neighbors of $v$ have label $c$:
$$
n_v(c;\sigma)
=
\sum_{u\in\mathcal N(v)}
\mathbf 1\{\sigma_u=c\}
$$

If $v$ changes from label $a$ to label $b$, only edges touching $v$ can change their conflict status.

Therefore
$$
\Delta H_v(a\rightarrow b)
=
H(\sigma^{v\rightarrow b})-H(\sigma)
$$
$$
\Delta H_v(a\rightarrow b)
=
J\left[n_v(b;\sigma)-n_v(a;\sigma)\right]
$$

$n_v(a;\sigma)$ is the number of old conflicts at $v$.

$n_v(b;\sigma)$ is the number of conflicts that the proposed label would create.

# 6. Metropolis Acceptance Function

Using the same notation as the example notebook, define the Metropolis ratio
$$
R_v(b\mid a)
=
\frac{
\pi(\sigma^{v\rightarrow b})Q_v(a\mid b)
}{
\pi(\sigma)Q_v(b\mid a)
}
=
e^{-\beta\Delta H_v(a\rightarrow b)}
$$

Because the proposal is symmetric, the acceptance probability is
$$
A_v(b\mid a)
=
\min\left(R_v(b\mid a),1\right)
$$

The probability of making the proposed one-vertex transition is
$$
T_v(\sigma^{v\rightarrow b}\mid\sigma)
=
Q_v(b\mid a)A_v(b\mid a)
$$

If the proposal lowers or keeps the energy,
$$
\Delta H_v\le 0
\quad\Rightarrow\quad
A_v=1
$$

If the proposal raises the energy,
$$
\Delta H_v>0
\quad\Rightarrow\quad
A_v=e^{-\beta\Delta H_v}
$$

The new label is therefore
$$
\sigma'_v
=
\begin{cases}
b, & \text{with probability }A_v(b\mid a),\\
a, & \text{with probability }1-A_v(b\mid a).
\end{cases}
$$

At a fixed $\beta$, this rule makes the one-vertex transition satisfy detailed balance with the Boltzmann distribution. Repeated sweeps converge to that distribution when the usual ergodicity conditions also hold.

# 7. Small Metropolis Example

Suppose
$$
q=3,
\qquad
J=1
$$

The current label $a$ has two matching neighbors:
$$
n_v(a;\sigma)=2
$$

If the proposed label $b$ has one matching neighbor, then
$$
n_v(b;\sigma)=1
$$
$$
\Delta H_v=1-2=-1
$$
$$
A_v=1
$$

This proposal is always accepted because it removes one conflict.

If another proposed label would create three conflicts, then
$$
\Delta H_v=3-2=1
$$
$$
A_v=e^{-\beta}
$$

For example, when $\beta=2$,
$$
A_v=e^{-2}\approx0.135
$$

# 8. Two Different Meanings of Color

This distinction is important.

**Potts label:**
$$
\sigma_v\in\{1,2,\ldots,q\}
$$

This is the color that our Markov chain changes while searching for a low-conflict graph coloring.

**Parallel schedule color:**
$$
c(v)\in\{1,2,\ldots,k\}
$$

This is fixed before sampling and is used only to decide which vertices may update at the same time.

The schedule must be a proper coloring:
$$
\{u,v\}\in E
\quad\Rightarrow\quad
c(u)\ne c(v)
$$

Define each parallel group by
$$
\kappa_r=\{v\in V:c(v)=r\}
$$

The numbers $q$ and $k$ do not have to be equal.

# 9. Why One Schedule Group Can Update in Parallel

No two vertices in the same schedule group are connected:
$$
u,v\in\kappa_r, u\ne v
\quad\Rightarrow\quad
\{u,v\}\notin E
$$

Therefore an update at $u$ does not change the local energy calculation at $v$ during the same group update.

All vertices in $\kappa_r$ calculate from the same state at the start of the group. Let $a_v=\sigma_v$, let $b_v$ be the proposed label, and define
$$
I_v
=
\begin{cases}
1, & \text{if the proposal at }v\text{ is accepted},\\
0, & \text{if it is rejected}.
\end{cases}
$$

The energy change that actually happens is
$$
H(\sigma')-H(\sigma)
=
\sum_{v\in\kappa_r}
I_vJ\left[n_v(b_v;\sigma)-n_v(a_v;\sigma)\right]
$$

The group transition is
$$
T_{\kappa_r}(\sigma'\mid\sigma)
=
\mathbf 1\!\left\{
\sigma'_{V\setminus\kappa_r}
=
\sigma_{V\setminus\kappa_r}
\right\}
\prod_{v\in\kappa_r}
T_v\!\left(
\sigma'_v\mid\sigma_v,\sigma_{\mathcal N(v)}
\right)
$$

Given the state at the start of the group, the local decisions are conditionally independent when their proposals and random accept/reject draws are independent.

It is not one large accept/reject decision for the whole group.

# 10. One Chromatic Parallel Sweep

Process the fixed schedule groups one at a time:
$$
\sigma^{(t,0)}
\xrightarrow{\ \kappa_1\text{ in parallel}\ }
\sigma^{(t,1)}
\xrightarrow{\ \kappa_2\text{ in parallel}\ }
\cdots
\xrightarrow{\ \kappa_k\text{ in parallel}\ }
\sigma^{(t+1,0)}
$$

Between two groups,
$$
\kappa_r\text{ in parallel}
\longrightarrow
\text{barrier}
\longrightarrow
\kappa_{r+1}\text{ in parallel}
$$

The barrier means every update in $\kappa_r$ finishes before $\kappa_{r+1}$ starts.

So later groups use the new labels produced by earlier groups.

The transition for one full sweep is
$$
T_{\mathrm{chrom}}
=
T_{\kappa_k}\circ T_{\kappa_{k-1}}\circ\cdots\circ T_{\kappa_1}
$$

The **chromatic** part is the schedule. The **Metropolis** part is the local proposal and acceptance function.

# 11. Small Parallel Schedule Example

For the path graph
$$
1-2-3-4-5-6
$$

a two-color parallel schedule is
$$
\kappa_1=\{1,3,5\}
$$
$$
\kappa_2=\{2,4,6\}
$$

One sweep is
$$
(1,3,5)\text{ update in parallel}
\longrightarrow
\text{barrier}
\longrightarrow
(2,4,6)\text{ update in parallel}
$$

The vertices in $\kappa_1$ are not neighbors of each other, and the same is true for $\kappa_2$.

Their changing Potts labels can still be any values in $\{1,\ldots,q\}$. The fixed schedule groups are not the graph-coloring solution.

# 12. Why the Base Chromatic Metropolis Chain Is Correct

For the symmetric one-vertex proposal, the Metropolis transition satisfies detailed balance:
$$
T_v(\sigma'\mid\sigma)
\pi(\sigma)
=
T_v(\sigma\mid\sigma')
\pi(\sigma')
$$

For two vertices $u$ and $v$ in the same schedule group,
$$
T_uT_v=T_vT_u
$$

because $u$ and $v$ are not connected and their local moves do not affect each other.

Each group transition preserves the Boltzmann distribution:
$$
\pi T_{\kappa_r}=\pi
$$

Therefore the complete chromatic sweep also preserves it:
$$
\pi T_{\mathrm{chrom}}=\pi
$$

These equations prove that $\pi$ is stationary. To also guarantee convergence from any starting coloring, the finite chain must be irreducible and aperiodic, and every vertex must continue to be updated.

Each one-vertex kernel and each same-group kernel is reversible. The complete deterministic sweep through different schedule groups may not be reversible, but it still preserves $\pi$ because it is a composition of invariant kernels.

This statement is for the **base Chromatic Metropolis chain at fixed $T>0$**. At $T=0$, uphill moves are impossible and the chain can become trapped. Lifting is a later change to the proposal direction and needs its own global-balance condition.

# 13. Parallel Running Time

Let
$$
n=|V|,
\qquad
p=\text{number of processors},
\qquad
k=\text{number of schedule groups}
$$

If each local update is treated as one unit of work, the number of parallel update slots is
$$
S_p
=
O\left(
\sum_{r=1}^{k}
\left\lceil\frac{|\kappa_r|}{p}\right\rceil
\right)
$$

Since
$$
\sum_{r=1}^{k}|\kappa_r|=n
$$

we get the scheduling bound
$$
S_p=O\left(\frac{n}{p}+k\right)
$$

The local conflict count at vertex $v$ requires
$$
O(\deg(v))
$$

The total conflict-counting work in one sweep is
$$
\sum_{v\in V}O(\deg(v))=O(|E|)
$$

An idealized degree-aware parallel bound is
$$
\mathrm{Time}_p
=
O\left(
\sum_{r=1}^{k}
\left[
\frac{\sum_{v\in\kappa_r}\deg(v)}{p}
+
\max_{v\in\kappa_r}\deg(v)
\right]
\right)
$$

The real time also includes barriers and scheduling overhead.

The $O(n/p+k)$ bound from Gonzalez et al. assumes that the schedule coloring is already available and that local update costs are similar. A greedy schedule coloring is enough for correctness; finding a coloring with the smallest possible $k$ is NP-hard.

Here the paper's scheduling structure is applied to our local Potts Metropolis updates.

# 14. Simulated Annealing in Our Proposal

Our proposal says that the temperature will decrease over time.

Keep one temperature $T_t$ fixed during the complete chromatic sweep $t$.

At step $t$,
$$
\beta_t=\frac{1}{T_t}
$$

The Boltzmann target at that temperature is
$$
\pi_t(\sigma)
\propto
e^{-H(\sigma)/T_t}
$$

The Metropolis ratio at sweep $t$ is
$$
R_{v,t}(b\mid a)
=
e^{-\Delta H_v(a\rightarrow b)/T_t}
$$

The Metropolis acceptance function becomes
$$
A_{v,t}(b\mid a)
=
\min\left(R_{v,t}(b\mid a),1\right)
$$

As $T_t$ decreases, an energy-increasing proposal is less likely to be accepted.

The chromatic schedule does not change: process one independent group in parallel, wait at the barrier, and then process the next group.

Because the temperature changes between sweeps, simulated annealing is a time-changing process and does not have one stationary distribution for the complete run.

A fast cooling schedule can trap the chain in a local minimum. Convergence to a global minimum is not automatically guaranteed for an arbitrary practical cooling schedule.

# 15. Evaluation Functions from Our Proposal

Energy after sweep $t$:
$$
H_t=H(\sigma^{(t)})
$$

Best energy reached by time $t$:
$$
H_{\min}(t)
=
\min_{0\le s\le t}H_s
$$

First time a valid coloring is found:
$$
\tau_0
=
\inf\{t\ge0:H_t=0\}
$$

$\tau_0$ can equal $+\infty$ when the graph is not $q$-colorable or when a finite run never reaches energy zero.

Energy autocorrelation at lag $\ell$:
$$
\rho_H(\ell)
=
\frac{\operatorname{Cov}(H_t,H_{t+\ell})}
{\operatorname{Var}(H_t)}
$$

Integrated autocorrelation time:
$$
\tau_{\mathrm{int}}
=
1+2\sum_{\ell=1}^{\infty}\rho_H(\ell)
$$

Approximate effective sample size from $N$ recorded samples:
$$
\mathrm{ESS}
\approx
\frac{N}{\tau_{\mathrm{int}}}
$$
$$
\mathrm{ESS/sec}
=
\frac{\mathrm{ESS}}{\text{runtime in seconds}}
$$

Autocorrelation and ESS should be measured after burn-in during a fixed-temperature stationary run, not across the changing-temperature optimization part of simulated annealing.

# 16. What Comes from the Paper and What Comes from Our Project

From Gonzalez, Low, Gretton, and Guestrin (2011), we use

- a proper coloring to form independent update groups,
- parallel updates inside one group,
- a barrier between groups,
- the scheduling time bound $O(n/p+k)$.

From our proposal, we use

- the antiferromagnetic $q$-state Potts conflict energy,
- the Boltzmann distribution,
- a local Metropolis proposal and acceptance function,
- simulated annealing,
- later addition of lifting.

The combination explained in this notebook is
$$
\text{chromatic schedule from the paper}
+
\text{Potts Metropolis local move from our proposal}
$$
$$
=
\text{Chromatic Parallel Metropolis}
$$

The paper's graph-coloring argument supplies the scheduling structure. The Metropolis stationarity argument for our Potts update is supplied separately in this notebook.

This notebook contains only the mathematics for the chromatic-parallel part. It does not contain implementation code.

---

**Project proposal:** *Graph Coloring Using Lifted Chromatic Metropolis Sampling for the Antiferromagnetic Potts Model*.

**Chromatic scheduling source:** Gonzalez, J. E., Low, Y., Gretton, A., and Guestrin, C. (2011). *Parallel Gibbs Sampling: From Colored Fields to Thin Junction Trees*. AISTATS, PMLR 15, 324-332.